# Interactive filtering interface for graph-based curriculum navigation

### An interactive, multi-criteria filtering interface generates a focus-centered (ego) subgraph from Neo4j and visualizes it to support concept lookup, constrained browsing by module/granularity/competence/phase, and discovery of co-occurring concepts.

In [4]:
# 1-Connect jupyter-notebook with neo4j

from neo4j import GraphDatabase

URI = "neo4j+s://c977fb54.databases.neo4j.io"
USER = "neo4j"
PASSWORD = "1ikLQEHSZT5SDMiXs_GFxnLBGAyNMNjOXPOrX94JOMg"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("✅ Connected to database")

# ============================================================
# ROBUST EGO NETWORK VIEW (Aura-safe, auto-reconnect) — FULL VERSION (ONE CELL)
#
# Fixes included:
#   ✅ Reads exported CSV/CYPHER (if present) + prints summary
#   ✅ Concept dropdown uses CSV concept list (ALL concepts), DB paging fallback
#   ✅ Focus node always matches dropdown selection; force-add focus node if missing
#   ✅ Safe cypher_df_driver keeps columns even when query returns 0 rows
#   ✅ NEW: highlights focus concept connections to OTHER concepts (via focus slides)
#       - background edges: light grey
#       - focus-touching edges: thick black
#       - focus-slide → other concept edges: red
#
# Requirements:
#   - driver already exists (neo4j.GraphDatabase.driver(...))
#   - pandas, networkx, plotly, ipywidgets installed
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from neo4j.exceptions import SessionExpired, ServiceUnavailable

# ------------------------------------------------------------
# 0) Paths to your exported files (choose ONE of the two blocks)
# ------------------------------------------------------------

BASE = Path("aura_runs_all_graph/run_20260210_180819/exports")
NODES_CSV  = BASE / "nodes_fetched.csv"
RELS_CSV   = BASE / "rels_fetched.csv"
CYPHER_TXT = BASE / "cypher_fetch_used.cypher"

# ------------------------------------------------------------
# 1) SAFE QUERY EXECUTION (AUTO-RECONNECT) — keeps columns even if 0 rows
# ------------------------------------------------------------
def cypher_df_driver(driver, query, params=None, retries=4, backoff=0.6):
    params = params or {}
    last_exc = None
    for attempt in range(retries + 1):
        try:
            with driver.session() as session:
                res = session.run(query, params)
                keys = list(res.keys())
                rows = [r.data() for r in res]
                return pd.DataFrame(rows, columns=keys)  # ✅ stable columns
        except (SessionExpired, ServiceUnavailable, OSError) as e:
            last_exc = e
            if attempt >= retries:
                raise
            time.sleep(backoff * (2 ** attempt))
    raise last_exc

def _to_bool(x):
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    s = str(x).strip().lower()
    return s in ("true", "1", "t", "yes", "y")

# ------------------------------------------------------------
# 2) READ EXPORTED FILES (nodes/edges + cypher used)
# ------------------------------------------------------------
def read_exports():
    nodes_df = None
    rels_df = None
    cypher_text = None

    if NODES_CSV.exists():
        nodes_df = pd.read_csv(NODES_CSV)
    if RELS_CSV.exists():
        rels_df = pd.read_csv(RELS_CSV)
    if CYPHER_TXT.exists():
        cypher_text = CYPHER_TXT.read_text(encoding="utf-8", errors="replace")

    return nodes_df, rels_df, cypher_text

def _normalize_exports(nodes_df, rels_df):
    if nodes_df is not None:
        node_cols = set(nodes_df.columns)
        ren_nodes = {}
        if "Id" in node_cols and "id" not in node_cols: ren_nodes["Id"] = "id"
        if "Label" in node_cols and "label" not in node_cols: ren_nodes["Label"] = "label"
        if "Name" in node_cols and "name" not in node_cols: ren_nodes["Name"] = "name"
        nodes_df = nodes_df.rename(columns=ren_nodes)

    if rels_df is not None:
        rel_cols = set(rels_df.columns)
        ren_rels = {}
        if "Source" in rel_cols and "source" not in rel_cols: ren_rels["Source"] = "source"
        if "Target" in rel_cols and "target" not in rel_cols: ren_rels["Target"] = "target"
        if "Type"   in rel_cols and "type"   not in rel_cols: ren_rels["Type"]   = "type"
        rels_df = rels_df.rename(columns=ren_rels)

    return nodes_df, rels_df

nodes_export, rels_export, cypher_used_text = read_exports()
nodes_export, rels_export = _normalize_exports(nodes_export, rels_export)

if nodes_export is not None and rels_export is not None and \
   ("id" in nodes_export.columns) and ("source" in rels_export.columns) and ("target" in rels_export.columns):

    nodes_u = nodes_export.dropna(subset=["id"]).drop_duplicates(subset=["id"]).copy()
    rels_u  = rels_export.dropna(subset=["source","target"]).copy()

    if "type" in rels_u.columns:
        rels_u = rels_u.drop_duplicates(subset=["source","target","type"])
    else:
        rels_u = rels_u.drop_duplicates(subset=["source","target"])

    print("=== Exported graph summary (from CSV) ===")
    print(f"Unique nodes: {nodes_u.shape[0]}")
    print(f"Unique edges: {rels_u.shape[0]}")
    if "label" in nodes_u.columns:
        print("\nNodes by label:")
        print(nodes_u["label"].astype(str).value_counts().sort_index())
    if "type" in rels_u.columns:
        print("\nEdges by type:")
        print(rels_u["type"].astype(str).value_counts().sort_index())
    print("========================================\n")

    if cypher_used_text is not None:
        print("Loaded cypher_fetch_used.cypher (first 400 chars):")
        print(cypher_used_text[:400])
        print("...\n")
else:
    print("⚠️ CSV exports not found or incomplete. Will rely on DB-only dropdown + ego query.\n")

# ------------------------------------------------------------
# 3) CONCEPT LIST SOURCE (CSV FIRST, then DB paging fallback)
# ------------------------------------------------------------
def concepts_from_csv(nodes_df):
    if nodes_df is None:
        return None
    if ("id" not in nodes_df.columns) or ("label" not in nodes_df.columns):
        return None
    df = nodes_df.dropna(subset=["id","label"]).copy()
    df["id"] = df["id"].astype(str)
    df["label"] = df["label"].astype(str)
    concepts = sorted(df.loc[df["label"] == "Concept", "id"].unique().tolist())
    return concepts if concepts else None

def fetch_all_concepts_from_db(driver):
    all_ids = []
    skip = 0
    page_size = 500
    while True:
        df = cypher_df_driver(driver, """
        MATCH (c:Concept)
        WHERE c.id IS NOT NULL
        RETURN DISTINCT toString(c.id) AS id
        ORDER BY id
        SKIP $skip LIMIT $limit
        """, {"skip": skip, "limit": page_size})

        if "id" not in df.columns:
            # query returned 0 rows + no keys (shouldn't happen now, but keep defensive)
            break

        batch = df["id"].dropna().astype(str).tolist()
        if not batch:
            break
        all_ids.extend(batch)
        skip += page_size

    seen = set()
    out = []
    for x in all_ids:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

ALL_CONCEPTS = concepts_from_csv(nodes_export)
if ALL_CONCEPTS is None:
    ALL_CONCEPTS = fetch_all_concepts_from_db(driver)

# ------------------------------------------------------------
# 4) CYPHER QUERY (NO UNION, NO APOC)
# ------------------------------------------------------------
EGO_EDGES_QUERY = """
MATCH (c:Concept)
OPTIONAL MATCH (c)-[:HAS_GRANULARITY]->(g0:Granularity)
OPTIONAL MATCH (c)-[:HAS_COMPETENCE]->(k0:Competence)
OPTIONAL MATCH (c)-[:HAS_PHASE]->(p0:Phase)
WHERE ($concept_id IS NULL OR $concept_id = "" OR c.id = $concept_id)
  AND ($granularity IS NULL OR $granularity = "" OR g0.id = $granularity)
  AND ($competence  IS NULL OR $competence  = "" OR k0.id = $competence)
  AND ($phase       IS NULL OR $phase       = "" OR p0.id = $phase)
WITH DISTINCT c AS c0

OPTIONAL MATCH (s:Slide)-[:FOCUSES_ON]->(c0)
OPTIONAL MATCH (m:Module)-[:HAS_SLIDE]->(s)
WITH c0, s, m
WHERE ($module IS NULL OR $module = "" OR m.id = $module)
WITH c0, collect(DISTINCT s)[0..$max_slides] AS slides
UNWIND slides AS s2
OPTIONAL MATCH (m2:Module)-[:HAS_SLIDE]->(s2)

MATCH (s2)-[:FOCUSES_ON]->(c2:Concept)
OPTIONAL MATCH (c2)-[:HAS_GRANULARITY]->(g2:Granularity)
OPTIONAL MATCH (c2)-[:HAS_COMPETENCE]->(k2:Competence)
OPTIONAL MATCH (c2)-[:HAS_PHASE]->(p2:Phase)
WHERE ($granularity IS NULL OR $granularity = "" OR g2.id = $granularity)
  AND ($competence  IS NULL OR $competence  = "" OR k2.id = $competence)
  AND ($phase       IS NULL OR $phase       = "" OR p2.id = $phase)

WITH c0, s2, m2, collect(DISTINCT c2) AS allc
WITH c0, s2, m2,
     [x IN allc WHERE x.id <> c0.id][0..$max_coconcepts_per_slide] + [c0] AS cocs
UNWIND cocs AS c_used

OPTIONAL MATCH (c_used)-[:HAS_GRANULARITY]->(gx:Granularity)
OPTIONAL MATCH (c_used)-[:HAS_COMPETENCE]->(kx:Competence)
OPTIONAL MATCH (c_used)-[:HAS_PHASE]->(px:Phase)

WITH c0, s2, m2, c_used, gx, kx, px,
[
  CASE WHEN m2 IS NULL OR s2 IS NULL THEN NULL ELSE {
    source:m2.id, sourceLabel:'Module', sourceName:coalesce(m2.name,m2.id), sourceFocus:false,
    target:s2.id, targetLabel:'Slide', targetName:coalesce(s2.name,s2.id), targetFocus:false,
    relType:'HAS_SLIDE'
  } END,

  CASE WHEN s2 IS NULL OR c_used IS NULL THEN NULL ELSE {
    source:s2.id, sourceLabel:'Slide', sourceName:coalesce(s2.name,s2.id), sourceFocus:false,
    target:c_used.id, targetLabel:'Concept', targetName:coalesce(c_used.name,c_used.id),
    targetFocus:(c_used.id = c0.id),
    relType:'FOCUSES_ON'
  } END,

  CASE WHEN gx IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:gx.id, targetLabel:'Granularity', targetName:coalesce(gx.name,gx.id), targetFocus:false,
    relType:'HAS_GRANULARITY'
  } END,

  CASE WHEN kx IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:kx.id, targetLabel:'Competence', targetName:coalesce(kx.name,kx.id), targetFocus:false,
    relType:'HAS_COMPETENCE'
  } END,

  CASE WHEN px IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:px.id, targetLabel:'Phase', targetName:coalesce(px.name,px.id), targetFocus:false,
    relType:'HAS_PHASE'
  } END
] AS edges

UNWIND edges AS e
WITH e WHERE e IS NOT NULL
RETURN
  e.source AS source, e.sourceLabel AS sourceLabel, e.sourceName AS sourceName, e.sourceFocus AS sourceFocus,
  e.target AS target, e.targetLabel AS targetLabel, e.targetName AS targetName, e.targetFocus AS targetFocus,
  e.relType AS relType
"""

FOCUS_NODE_QUERY = """
MATCH (c:Concept {id: $id})
RETURN
  toString(c.id) AS id,
  'Concept' AS label,
  coalesce(c.name, toString(c.id)) AS name,
  true AS is_focus
LIMIT 1
"""

# ------------------------------------------------------------
# 5) FETCH TABLES (force-add focus node if missing)
# ------------------------------------------------------------
def fetch_ego(concept_id="", module="", granularity="", competence="", phase="",
              max_slides=60, max_coconcepts_per_slide=80):

    concept_id = (concept_id or "").strip()

    df = cypher_df_driver(driver, EGO_EDGES_QUERY, {
        "concept_id": concept_id,
        "module": (module or ""),
        "granularity": (granularity or ""),
        "competence": (competence or ""),
        "phase": (phase or ""),
        "max_slides": int(max_slides),
        "max_coconcepts_per_slide": int(max_coconcepts_per_slide),
    })

    if df.empty:
        nodes = pd.DataFrame(columns=["id","label","name","is_focus"])
        rels  = pd.DataFrame(columns=["source","target","type"])
    else:
        left = df.rename(columns={
            "source":"id","sourceLabel":"label","sourceName":"name","sourceFocus":"is_focus"
        })[["id","label","name","is_focus"]]

        right = df.rename(columns={
            "target":"id","targetLabel":"label","targetName":"name","targetFocus":"is_focus"
        })[["id","label","name","is_focus"]]

        nodes = pd.concat([left, right], ignore_index=True).dropna(subset=["id"]).drop_duplicates("id")
        nodes["id"] = nodes["id"].astype(str)
        nodes["label"] = nodes["label"].astype(str)
        nodes["name"] = nodes["name"].astype(str)
        nodes["is_focus"] = nodes["is_focus"].apply(_to_bool)

        rels = df.rename(columns={"relType":"type"})[["source","target","type"]].dropna()
        rels["source"] = rels["source"].astype(str)
        rels["target"] = rels["target"].astype(str)
        rels["type"] = rels["type"].astype(str)

    if concept_id:
        if nodes.empty or concept_id not in set(nodes["id"].astype(str)):
            fdf = cypher_df_driver(driver, FOCUS_NODE_QUERY, {"id": concept_id})
            if not fdf.empty and "id" in fdf.columns:
                fdf["id"] = fdf["id"].astype(str)
                fdf["label"] = fdf["label"].astype(str)
                fdf["name"] = fdf["name"].astype(str)
                fdf["is_focus"] = fdf["is_focus"].apply(_to_bool)
                nodes = pd.concat([nodes, fdf[["id","label","name","is_focus"]]],
                                  ignore_index=True).drop_duplicates("id")

    return nodes, rels

# ------------------------------------------------------------
# 6) PLOT (reliable coconcept highlighting from rels dataframe)
# ------------------------------------------------------------
NODE_COLORS = {
    "Concept":"#1F77B4",
    "Module":"#2A9D8F",
    "Slide":"#6C757D",
    "Phase":"#D1495B",
    "Competence":"#E9C46A",
    "Granularity":"#2CA02C",
    "Other":"#B0B0B0",
}
FOCUS_NODE_COLOR = "#7C3AED"
FOCUS_RING_COLOR = "#FBBF24"
COCONCEPT_EDGE_COLOR = "#EF4444"  # red

def plot_graph(nodes, rels, focus_id="", title="Ego Network (focus centered)"):
    focus_id = (focus_id or "").strip()

    if nodes.empty:
        fig = go.Figure()
        fig.update_layout(title="No results (try relaxing filters)")
        return fig

    # label lookup
    node_label = {}
    for r in nodes.to_dict("records"):
        nid = str(r["id"])
        lab = str(r.get("label", "Other")).strip()
        node_label[nid] = lab

    # graph for layout only
    G = nx.Graph()
    for r in nodes.to_dict("records"):
        G.add_node(str(r["id"]), **r)

    for r in rels.to_dict("records"):
        u = str(r["source"]); v = str(r["target"])
        G.add_edge(u, v, type=str(r.get("type", "")))

    focus = focus_id if (focus_id and focus_id in G.nodes) else None

    init_pos = nx.random_layout(G, seed=42)
    if focus:
        init_pos[focus] = np.array([0.0, 0.0])

    pos = nx.spring_layout(
        G, seed=42,
        pos=init_pos,
        fixed=[focus] if focus else None,
        k=2.2 / max(1, (G.number_of_nodes() ** 0.5)),
        iterations=400
    )

    if focus and focus in pos:
        fx, fy = pos[focus]
        for n in pos:
            pos[n] = (pos[n][0] - fx, pos[n][1] - fy)

    # ---- focus slides + coconcept edges FROM rels (robust)
    focus_slides = set()
    coc_edges = set()

    if focus and (rels is not None) and (not rels.empty):
        r2 = rels.copy()
        r2["source"] = r2["source"].astype(str)
        r2["target"] = r2["target"].astype(str)
        r2["type"] = r2["type"].astype(str)

        foc_rel = r2[r2["type"] == "FOCUSES_ON"]

        for _, row in foc_rel.iterrows():
            u, v = row["source"], row["target"]
            if u == focus and node_label.get(v, "") == "Slide":
                focus_slides.add(v)
            elif v == focus and node_label.get(u, "") == "Slide":
                focus_slides.add(u)

        if focus_slides:
            for _, row in foc_rel.iterrows():
                u, v = row["source"], row["target"]
                # slide -> concept (either direction), exclude focus
                if u in focus_slides and node_label.get(v, "") == "Concept" and v != focus:
                    coc_edges.add((u, v))
                elif v in focus_slides and node_label.get(u, "") == "Concept" and u != focus:
                    coc_edges.add((v, u))

    # ---- Edge buckets
    bg_x, bg_y = [], []
    focus_x, focus_y = [], []
    coc_x, coc_y = [], []

    def _seg(arrx, arry, u, v):
        x0, y0 = pos[u]; x1, y1 = pos[v]
        arrx += [x0, x1, None]
        arry += [y0, y1, None]

    for u, v in G.edges():
        u = str(u); v = str(v)
        is_focus_edge = bool(focus) and (u == focus or v == focus)
        is_coconcept_edge = ((u, v) in coc_edges) or ((v, u) in coc_edges)

        if is_coconcept_edge:
            _seg(coc_x, coc_y, u, v)
        elif is_focus_edge:
            _seg(focus_x, focus_y, u, v)
        else:
            _seg(bg_x, bg_y, u, v)

    edge_bg = go.Scatter(
        x=bg_x, y=bg_y, mode="lines",
        line=dict(width=1, color="#B8BCC2"),
        opacity=0.18, hoverinfo="none",
        name="edges"
    )
    edge_coconcepts = go.Scatter(
        x=coc_x, y=coc_y, mode="lines",
        line=dict(width=2.8, color=COCONCEPT_EDGE_COLOR),
        opacity=0.90, hoverinfo="none",
        name="edges (focus slides → other concepts)"
    )
    edge_focus = go.Scatter(
        x=focus_x, y=focus_y, mode="lines",
        line=dict(width=3.2, color="#111827"),
        opacity=0.85, hoverinfo="none",
        name="edges (focus)"
    )

    # ---- Nodes except focus
    node_traces = []
    by_label = {}
    for n, d in G.nodes(data=True):
        if focus and n == focus:
            continue
        by_label.setdefault(str(d.get("label", "Other")).strip(), []).append((n, d))

    for lab, items in sorted(by_label.items()):
        xs, ys, h = [], [], []
        sizes, cols = [], []
        for n, d in items:
            x, y = pos[n]
            xs.append(x); ys.append(y)
            h.append(
                "<br>".join([
                    f"<b>{d.get('id', n)}</b>",
                    f"type: {d.get('label', '')}",
                    f"name: {d.get('name', '')}",
                ])
            )
            sizes.append(14 if lab == "Concept" else 10)
            cols.append(NODE_COLORS.get(lab, NODE_COLORS["Other"]))

        node_traces.append(go.Scatter(
            x=xs, y=ys, mode="markers",
            marker=dict(size=sizes, color=cols, opacity=0.95, line=dict(width=0)),
            hovertext=h, hoverinfo="text",
            name=f"Node: {lab}",
            showlegend=True
        ))

    # ---- Focus marker
    focus_trace = []
    warning = ""
    if focus_id and (focus is None):
        warning = "⚠️ Selected concept is not present under current filters (try relaxing module/granularity/competence/phase)."
    elif focus:
        fx, fy = pos[focus]
        fd = G.nodes[focus]
        label = str(fd.get("id", focus))
        if len(label) > 70:
            label = label[:67] + "..."

        focus_trace = [
            go.Scatter(
                x=[fx], y=[fy], mode="markers",
                marker=dict(size=70, color=FOCUS_NODE_COLOR, opacity=1.0,
                            line=dict(width=7, color=FOCUS_RING_COLOR)),
                hovertext=[f"<b>{fd.get('id', focus)}</b><br>FOCUS (selected concept)"],
                hoverinfo="text",
                name="FOCUS (selected concept)",
                showlegend=True
            ),
            go.Scatter(
                x=[fx], y=[fy], mode="text",
                text=[label],
                textposition="middle right",
                textfont=dict(size=14),
                hoverinfo="none",
                showlegend=False
            )
        ]

    fig = go.Figure([edge_bg, edge_coconcepts, edge_focus] + node_traces + focus_trace)
    fig.update_layout(
        title=f"{title} — {G.number_of_nodes()} nodes / {G.number_of_edges()} edges",
        hovermode="closest",
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(itemsizing="constant"),
    )

    if warning:
        fig.add_annotation(
            xref="paper", yref="paper",
            x=0.01, y=1.08,
            text=warning,
            showarrow=False,
            font=dict(size=12),
            bgcolor="rgba(255,255,255,0.85)"
        )

    return fig
def plot_graph(nodes, rels, focus_id="", title="Ego Network (focus centered)"):
    focus_id = (focus_id or "").strip()

    if nodes.empty:
        fig = go.Figure()
        fig.update_layout(title="No results (try relaxing filters)")
        return fig

    # label lookup
    node_label = {}
    for r in nodes.to_dict("records"):
        nid = str(r["id"])
        lab = str(r.get("label", "Other")).strip()
        node_label[nid] = lab

    # graph for layout only
    G = nx.Graph()
    for r in nodes.to_dict("records"):
        G.add_node(str(r["id"]), **r)

    for r in rels.to_dict("records"):
        u = str(r["source"]); v = str(r["target"])
        G.add_edge(u, v, type=str(r.get("type", "")))

    focus = focus_id if (focus_id and focus_id in G.nodes) else None

    init_pos = nx.random_layout(G, seed=42)
    if focus:
        init_pos[focus] = np.array([0.0, 0.0])

    pos = nx.spring_layout(
        G, seed=42,
        pos=init_pos,
        fixed=[focus] if focus else None,
        k=2.2 / max(1, (G.number_of_nodes() ** 0.5)),
        iterations=400
    )

    if focus and focus in pos:
        fx, fy = pos[focus]
        for n in pos:
            pos[n] = (pos[n][0] - fx, pos[n][1] - fy)

    # -----------------------------
    # Build "focus slides" and "co-concepts"
    # -----------------------------
    focus_slides = set()
    coc_edges_slide_to_concept = set()
    coc_concepts = {}  # concept_id -> count of shared focus-slides (for optional thickness)

    if focus and (rels is not None) and (not rels.empty):
        r2 = rels.copy()
        r2["source"] = r2["source"].astype(str)
        r2["target"] = r2["target"].astype(str)
        r2["type"] = r2["type"].astype(str)

        foc_rel = r2[r2["type"] == "FOCUSES_ON"]

        # Find slides that are connected to the focus concept
        for _, row in foc_rel.iterrows():
            u, v = row["source"], row["target"]
            if u == focus and node_label.get(v, "") == "Slide":
                focus_slides.add(v)
            elif v == focus and node_label.get(u, "") == "Slide":
                focus_slides.add(u)

        # On those slides, find all OTHER concepts
        if focus_slides:
            for _, row in foc_rel.iterrows():
                u, v = row["source"], row["target"]

                # slide -> concept (either direction), exclude focus
                if u in focus_slides and node_label.get(v, "") == "Concept" and v != focus:
                    coc_edges_slide_to_concept.add((u, v))
                    coc_concepts[v] = coc_concepts.get(v, 0) + 1
                elif v in focus_slides and node_label.get(u, "") == "Concept" and u != focus:
                    coc_edges_slide_to_concept.add((v, u))
                    coc_concepts[u] = coc_concepts.get(u, 0) + 1

    # -----------------------------
    # Edge buckets (real edges)
    # -----------------------------
    bg_x, bg_y = [], []
    focus_x, focus_y = [], []
    coc_slide_x, coc_slide_y = [], []   # Slide -> other concept edges (optional)
    coc_virtual_x, coc_virtual_y = [], []  # NEW: Focus -> other concept virtual edges

    def _seg(arrx, arry, u, v):
        x0, y0 = pos[u]; x1, y1 = pos[v]
        arrx += [x0, x1, None]
        arry += [y0, y1, None]

    # Real edges
    for u, v in G.edges():
        u = str(u); v = str(v)
        is_focus_edge = bool(focus) and (u == focus or v == focus)
        is_coconcept_slide_edge = ((u, v) in coc_edges_slide_to_concept) or ((v, u) in coc_edges_slide_to_concept)

        if is_coconcept_slide_edge:
            _seg(coc_slide_x, coc_slide_y, u, v)
        elif is_focus_edge:
            _seg(focus_x, focus_y, u, v)
        else:
            _seg(bg_x, bg_y, u, v)

    # Virtual edges: focus -> each co-concept
    if focus:
        for c_id in coc_concepts.keys():
            if c_id in pos:
                _seg(coc_virtual_x, coc_virtual_y, focus, c_id)

    edge_bg = go.Scatter(
        x=bg_x, y=bg_y, mode="lines",
        line=dict(width=1, color="#B8BCC2"),
        opacity=0.18, hoverinfo="none",
        name="edges"
    )

    # Optional: keep slide->concept highlight (can remove if you want)
    edge_coc_slides = go.Scatter(
        x=coc_slide_x, y=coc_slide_y, mode="lines",
        line=dict(width=2.0, color=COCONCEPT_EDGE_COLOR),
        opacity=0.35, hoverinfo="none",
        name="edges (focus slides → other concepts)"
    )

    # ✅ This is what you asked: purple concept connected directly to other blue concepts
    edge_coc_virtual = go.Scatter(
        x=coc_virtual_x, y=coc_virtual_y, mode="lines",
        line=dict(width=3.6, color=COCONCEPT_EDGE_COLOR),
        opacity=0.95, hoverinfo="none",
        name="edges (FOCUS → other concepts)"
    )

    edge_focus = go.Scatter(
        x=focus_x, y=focus_y, mode="lines",
        line=dict(width=3.2, color="#111827"),
        opacity=0.85, hoverinfo="none",
        name="edges (focus)"
    )

    # -----------------------------
    # Nodes (grouped by label)
    # -----------------------------
    node_traces = []
    by_label = {}
    for n, d in G.nodes(data=True):
        if focus and n == focus:
            continue
        by_label.setdefault(str(d.get("label", "Other")).strip(), []).append((n, d))

    for lab, items in sorted(by_label.items()):
        xs, ys, h = [], [], []
        sizes, cols = [], []
        for n, d in items:
            x, y = pos[n]
            xs.append(x); ys.append(y)
            h.append(
                "<br>".join([
                    f"<b>{d.get('id', n)}</b>",
                    f"type: {d.get('label', '')}",
                    f"name: {d.get('name', '')}",
                ])
            )
            sizes.append(14 if lab == "Concept" else 10)
            cols.append(NODE_COLORS.get(lab, NODE_COLORS["Other"]))

        node_traces.append(go.Scatter(
            x=xs, y=ys, mode="markers",
            marker=dict(size=sizes, color=cols, opacity=0.95, line=dict(width=0)),
            hovertext=h, hoverinfo="text",
            name=f"Node: {lab}",
            showlegend=True
        ))

    # -----------------------------
    # Focus marker
    # -----------------------------
    focus_trace = []
    warning = ""
    if focus_id and (focus is None):
        warning = "⚠️ Selected concept is not present under current filters (try relaxing module/granularity/competence/phase)."
    elif focus:
        fx, fy = pos[focus]
        fd = G.nodes[focus]
        label = str(fd.get("id", focus))
        if len(label) > 70:
            label = label[:67] + "..."

        focus_trace = [
            go.Scatter(
                x=[fx], y=[fy], mode="markers",
                marker=dict(size=70, color=FOCUS_NODE_COLOR, opacity=1.0,
                            line=dict(width=7, color=FOCUS_RING_COLOR)),
                hovertext=[f"<b>{fd.get('id', focus)}</b><br>FOCUS (selected concept)"],
                hoverinfo="text",
                name="FOCUS (selected concept)",
                showlegend=True
            ),
            go.Scatter(
                x=[fx], y=[fy], mode="text",
                text=[label],
                textposition="middle right",
                textfont=dict(size=14),
                hoverinfo="none",
                showlegend=False
            )
        ]

    # Order: background -> slide highlights -> virtual focus->concept -> focus edges -> nodes -> focus marker
    fig = go.Figure([edge_bg, edge_coc_slides, edge_coc_virtual, edge_focus] + node_traces + focus_trace)
    fig.update_layout(
        title=f"{title} — {G.number_of_nodes()} nodes / {G.number_of_edges()} edges",
        hovermode="closest",
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(itemsizing="constant"),
    )

    if warning:
        fig.add_annotation(
            xref="paper", yref="paper",
            x=0.01, y=1.08,
            text=warning,
            showarrow=False,
            font=dict(size=12),
            bgcolor="rgba(255,255,255,0.85)"
        )

    return fig

# ------------------------------------------------------------
# 7) UI (Module/Granularity/Competence/Phase from DB; Concepts from CSV)
# ------------------------------------------------------------
modules = cypher_df_driver(driver, "MATCH (m:Module) RETURN DISTINCT toString(m.id) AS id ORDER BY id")["id"].dropna().tolist()
grans   = cypher_df_driver(driver, "MATCH (g:Granularity) RETURN DISTINCT toString(g.id) AS id ORDER BY id")["id"].dropna().tolist()
comps   = cypher_df_driver(driver, "MATCH (k:Competence) RETURN DISTINCT toString(k.id) AS id ORDER BY id")["id"].dropna().tolist()
phases  = cypher_df_driver(driver, "MATCH (p:Phase) RETURN DISTINCT toString(p.id) AS id ORDER BY id")["id"].dropna().tolist()

module_dd = widgets.Dropdown(options=[""] + modules, description="Module:", layout=widgets.Layout(width="240px"))
gran_dd   = widgets.Dropdown(options=[""] + grans,   description="Granularity:", layout=widgets.Layout(width="240px"))
comp_dd   = widgets.Dropdown(options=[""] + comps,   description="Competence:", layout=widgets.Layout(width="240px"))
phase_dd  = widgets.Dropdown(options=[""] + phases,  description="Phase:", layout=widgets.Layout(width="240px"))

concept_search = widgets.Text(value="", description="Search:", layout=widgets.Layout(width="420px"))
concept_dd = widgets.Dropdown(options=[""] + ALL_CONCEPTS, description="Concept:", layout=widgets.Layout(width="520px"))

slides_sl  = widgets.IntSlider(value=60, min=5, max=400, step=5, description="Max slides:", layout=widgets.Layout(width="320px"))
co_sl      = widgets.IntSlider(value=80, min=1, max=800, step=1, description="Co-concepts/slide:", layout=widgets.Layout(width="360px"))

apply_btn  = widgets.Button(description="Apply", button_style="primary", layout=widgets.Layout(width="120px"))
clear_btn  = widgets.Button(description="Clear", layout=widgets.Layout(width="120px"))
status_lbl = widgets.HTML(value="")
out = widgets.Output()

def update_concept_options(*_):
    q = concept_search.value.strip().lower()
    selected = (concept_dd.value or "").strip()

    if not q:
        opts = [""] + ALL_CONCEPTS
    else:
        filtered = [c for c in ALL_CONCEPTS if q in c.lower()]
        if selected and (selected not in filtered):
            filtered = [selected] + filtered
        opts = [""] + filtered

    concept_dd.options = opts
    if selected and selected not in concept_dd.options:
        concept_dd.value = ""

def refresh(_=None):
    with out:
        clear_output(wait=True)

        status_lbl.value = (
            f"<b>Ego view:</b> concept='{concept_dd.value or '*'}', "
            f"module='{module_dd.value or '*'}', "
            f"granularity='{gran_dd.value or '*'}', "
            f"competence='{comp_dd.value or '*'}', "
            f"phase='{phase_dd.value or '*'}', "
            f"max_slides={slides_sl.value}, "
            f"co_concepts/slide={co_sl.value}"
        )

        nodes, rels = fetch_ego(
            concept_id=concept_dd.value,
            module=module_dd.value,
            granularity=gran_dd.value,
            competence=comp_dd.value,
            phase=phase_dd.value,
            max_slides=slides_sl.value,
            max_coconcepts_per_slide=co_sl.value
        )

        fig = plot_graph(nodes, rels, focus_id=concept_dd.value, title="Ego Network (focus centered)")
        fig.show()

def clear_filters(_=None):
    module_dd.value = ""
    gran_dd.value = ""
    comp_dd.value = ""
    phase_dd.value = ""
    concept_search.value = ""
    concept_dd.value = ""
    slides_sl.value = 60
    co_sl.value = 80
    refresh()

concept_search.observe(update_concept_options, names="value")
apply_btn.on_click(refresh)
clear_btn.on_click(clear_filters)

display(widgets.VBox([
    widgets.HBox([module_dd, gran_dd, comp_dd, phase_dd]),
    widgets.HBox([concept_search, concept_dd]),
    widgets.HBox([slides_sl, co_sl, apply_btn, clear_btn]),
    status_lbl,
    out
]))

update_concept_options()
refresh()


✅ Connected to database
=== Exported graph summary (from CSV) ===
Unique nodes: 568
Unique edges: 1332

Nodes by label:
label
Competence       3
Concept        236
Granularity      3
Module          10
Phase            4
Slide          312
Name: count, dtype: int64

Edges by type:
type
FOCUSES_ON         312
HAS_COMPETENCE     236
HAS_GRANULARITY    236
HAS_PHASE          236
HAS_SLIDE          312
Name: count, dtype: int64

Loaded cypher_fetch_used.cypher (first 400 chars):
// QUERY 1: FETCH_NODES_ALL
MATCH (n)
WHERE n.id IS NOT NULL
RETURN
  n.id AS id,
  labels(n)[0] AS label,
  coalesce(n.name, n.id) AS name,
  coalesce(n.frequency, 0) AS frequency,
  coalesce(n.modules, 0) AS modules,
  coalesce(n.phase_mode, "") AS phase_mode,
  coalesce(n.competence_mode, "") AS competence_mode,
  coalesce(n.granularity_mode, "") AS granularity_mode

// QUERY 2: FETCH_EDGES_ALL
...



In [5]:

# ============================================================
# ROBUST EGO NETWORK VIEW (Aura-safe, auto-reconnect) — FULL VERSION (ONE CELL)
#
# Adds what you want:
# ✅ Keeps ALL existing real edges (Module–Slide, Slide–Concept, Concept–{Phase/Competence/Granularity})
# ✅ ALSO adds a derived "CO_OCCURS_WITH" edge between:
#       focus Concept  →  other Concept
#   when they co-occur on the same focus slide(s).
#
# How it works:
# - Cypher returns the real graph edges as before
# - PLUS a derived edge: Concept(focus) -[:CO_OCCURS_WITH]-> Concept(other)
# - Plotly draws:
#     background edges (light grey)
#     focus-touching edges (thick black)
#     focus-slide → other concept edges (red, optional)
#     focus → other concept CO_OCCURS_WITH edges (red, thick)
#
# Requirements:
# - driver already exists (neo4j.GraphDatabase.driver(...))
# - pandas, networkx, plotly, ipywidgets installed
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from neo4j.exceptions import SessionExpired, ServiceUnavailable

# ------------------------------------------------------------
# 0) Paths to your exported files (optional; used for concept list)
# ------------------------------------------------------------

BASE = Path("aura_runs_all_graph/run_20260210_180819/exports")
NODES_CSV  = BASE / "nodes_fetched.csv"
RELS_CSV   = BASE / "rels_fetched.csv"
CYPHER_TXT = BASE / "cypher_fetch_used.cypher"

# Option B (your run folder):
# BASE = Path("aura_runs_all_graph/run_20260210_180819/exports")
# NODES_CSV  = BASE / "nodes_fetched.csv"
# RELS_CSV   = BASE / "rels_fetched.csv"
# CYPHER_TXT = BASE / "cypher_fetch_used.cypher"

# ------------------------------------------------------------
# 1) SAFE QUERY EXECUTION (AUTO-RECONNECT) — keeps columns even if 0 rows
# ------------------------------------------------------------
def cypher_df_driver(driver, query, params=None, retries=4, backoff=0.6):
    params = params or {}
    last_exc = None
    for attempt in range(retries + 1):
        try:
            with driver.session() as session:
                res = session.run(query, params)
                keys = list(res.keys())
                rows = [r.data() for r in res]
                return pd.DataFrame(rows, columns=keys)  # stable columns
        except (SessionExpired, ServiceUnavailable, OSError) as e:
            last_exc = e
            if attempt >= retries:
                raise
            time.sleep(backoff * (2 ** attempt))
    raise last_exc

def _to_bool(x):
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    s = str(x).strip().lower()
    return s in ("true", "1", "t", "yes", "y")

# ------------------------------------------------------------
# 2) READ EXPORTED FILES (nodes/edges + cypher used) — optional
# ------------------------------------------------------------
def read_exports():
    nodes_df = None
    rels_df = None
    cypher_text = None

    if NODES_CSV.exists():
        nodes_df = pd.read_csv(NODES_CSV)
    if RELS_CSV.exists():
        rels_df = pd.read_csv(RELS_CSV)
    if CYPHER_TXT.exists():
        cypher_text = CYPHER_TXT.read_text(encoding="utf-8", errors="replace")

    return nodes_df, rels_df, cypher_text

def _normalize_exports(nodes_df, rels_df):
    if nodes_df is not None:
        node_cols = set(nodes_df.columns)
        ren_nodes = {}
        if "Id" in node_cols and "id" not in node_cols: ren_nodes["Id"] = "id"
        if "Label" in node_cols and "label" not in node_cols: ren_nodes["Label"] = "label"
        if "Name" in node_cols and "name" not in node_cols: ren_nodes["Name"] = "name"
        nodes_df = nodes_df.rename(columns=ren_nodes)

    if rels_df is not None:
        rel_cols = set(rels_df.columns)
        ren_rels = {}
        if "Source" in rel_cols and "source" not in rel_cols: ren_rels["Source"] = "source"
        if "Target" in rel_cols and "target" not in rel_cols: ren_rels["Target"] = "target"
        if "Type"   in rel_cols and "type"   not in rel_cols: ren_rels["Type"]   = "type"
        rels_df = rels_df.rename(columns=ren_rels)

    return nodes_df, rels_df

nodes_export, rels_export, cypher_used_text = read_exports()
nodes_export, rels_export = _normalize_exports(nodes_export, rels_export)

if nodes_export is not None and rels_export is not None and \
   ("id" in nodes_export.columns) and ("source" in rels_export.columns) and ("target" in rels_export.columns):

    nodes_u = nodes_export.dropna(subset=["id"]).drop_duplicates(subset=["id"]).copy()
    rels_u  = rels_export.dropna(subset=["source","target"]).copy()

    if "type" in rels_u.columns:
        rels_u = rels_u.drop_duplicates(subset=["source","target","type"])
    else:
        rels_u = rels_u.drop_duplicates(subset=["source","target"])

    print("=== Exported graph summary (from CSV) ===")
    print(f"Unique nodes: {nodes_u.shape[0]}")
    print(f"Unique edges: {rels_u.shape[0]}")
    if "label" in nodes_u.columns:
        print("\nNodes by label:")
        print(nodes_u["label"].astype(str).value_counts().sort_index())
    if "type" in rels_u.columns:
        print("\nEdges by type:")
        print(rels_u["type"].astype(str).value_counts().sort_index())
    print("========================================\n")

    if cypher_used_text is not None:
        print("Loaded cypher_fetch_used.cypher (first 400 chars):")
        print(cypher_used_text[:400])
        print("...\n")
else:
    print("⚠️ CSV exports not found or incomplete. Will rely on DB-only dropdown + ego query.\n")

# ------------------------------------------------------------
# 3) CONCEPT LIST SOURCE (CSV FIRST, then DB paging fallback)
# ------------------------------------------------------------
def concepts_from_csv(nodes_df):
    if nodes_df is None:
        return None
    if ("id" not in nodes_df.columns) or ("label" not in nodes_df.columns):
        return None
    df = nodes_df.dropna(subset=["id","label"]).copy()
    df["id"] = df["id"].astype(str)
    df["label"] = df["label"].astype(str)
    concepts = sorted(df.loc[df["label"].str.strip().str.lower() == "concept", "id"].unique().tolist())
    return concepts if concepts else None

def fetch_all_concepts_from_db(driver):
    all_ids = []
    skip = 0
    page_size = 500
    while True:
        df = cypher_df_driver(driver, """
        MATCH (c:Concept)
        WHERE c.id IS NOT NULL
        RETURN DISTINCT toString(c.id) AS id
        ORDER BY id
        SKIP $skip LIMIT $limit
        """, {"skip": skip, "limit": page_size})

        if "id" not in df.columns:
            break

        batch = df["id"].dropna().astype(str).tolist()
        if not batch:
            break
        all_ids.extend(batch)
        skip += page_size

    seen = set()
    out = []
    for x in all_ids:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

ALL_CONCEPTS = concepts_from_csv(nodes_export)
if ALL_CONCEPTS is None:
    ALL_CONCEPTS = fetch_all_concepts_from_db(driver)

# ------------------------------------------------------------
# 4) CYPHER QUERY (NO UNION, NO APOC)
#    ✅ Adds derived CO_OCCURS_WITH edges (focus -> other concepts)
# ------------------------------------------------------------
EGO_EDGES_QUERY = """
MATCH (c:Concept)
OPTIONAL MATCH (c)-[:HAS_GRANULARITY]->(g0:Granularity)
OPTIONAL MATCH (c)-[:HAS_COMPETENCE]->(k0:Competence)
OPTIONAL MATCH (c)-[:HAS_PHASE]->(p0:Phase)
WHERE ($concept_id IS NULL OR $concept_id = "" OR c.id = $concept_id)
  AND ($granularity IS NULL OR $granularity = "" OR g0.id = $granularity)
  AND ($competence  IS NULL OR $competence  = "" OR k0.id = $competence)
  AND ($phase       IS NULL OR $phase       = "" OR p0.id = $phase)
WITH DISTINCT c AS c0

// slides that focus on the selected concept
OPTIONAL MATCH (s:Slide)-[:FOCUSES_ON]->(c0)
OPTIONAL MATCH (m:Module)-[:HAS_SLIDE]->(s)
WITH c0, s, m
WHERE ($module IS NULL OR $module = "" OR m.id = $module)
WITH c0, collect(DISTINCT s)[0..$max_slides] AS slides
UNWIND slides AS s2
OPTIONAL MATCH (m2:Module)-[:HAS_SLIDE]->(s2)

// all concepts on each focus slide
MATCH (s2)-[:FOCUSES_ON]->(c2:Concept)
OPTIONAL MATCH (c2)-[:HAS_GRANULARITY]->(g2:Granularity)
OPTIONAL MATCH (c2)-[:HAS_COMPETENCE]->(k2:Competence)
OPTIONAL MATCH (c2)-[:HAS_PHASE]->(p2:Phase)
WHERE ($granularity IS NULL OR $granularity = "" OR g2.id = $granularity)
  AND ($competence  IS NULL OR $competence  = "" OR k2.id = $competence)
  AND ($phase       IS NULL OR $phase       = "" OR p2.id = $phase)

WITH c0, s2, m2, collect(DISTINCT c2) AS allc
WITH c0, s2, m2,
     [x IN allc WHERE x.id <> c0.id][0..$max_coconcepts_per_slide] AS other_concepts,
     [x IN allc][0..$max_coconcepts_per_slide] AS concepts_for_edges

// --- build rows for BOTH:
//     (A) "real" edges via slide (module->slide, slide->concept)
//     (B) "derived" focus->other concept edges: CO_OCCURS_WITH
UNWIND concepts_for_edges AS c_used

OPTIONAL MATCH (c_used)-[:HAS_GRANULARITY]->(gx:Granularity)
OPTIONAL MATCH (c_used)-[:HAS_COMPETENCE]->(kx:Competence)
OPTIONAL MATCH (c_used)-[:HAS_PHASE]->(px:Phase)

WITH c0, s2, m2, c_used, gx, kx, px, other_concepts,
[
  CASE WHEN m2 IS NULL OR s2 IS NULL THEN NULL ELSE {
    source:m2.id, sourceLabel:'Module', sourceName:coalesce(m2.name,m2.id), sourceFocus:false,
    target:s2.id, targetLabel:'Slide', targetName:coalesce(s2.name,s2.id), targetFocus:false,
    relType:'HAS_SLIDE'
  } END,

  CASE WHEN s2 IS NULL OR c_used IS NULL THEN NULL ELSE {
    source:s2.id, sourceLabel:'Slide', sourceName:coalesce(s2.name,s2.id), sourceFocus:false,
    target:c_used.id, targetLabel:'Concept', targetName:coalesce(c_used.name,c_used.id),
    targetFocus:(c_used.id = c0.id),
    relType:'FOCUSES_ON'
  } END,

  CASE WHEN gx IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:gx.id, targetLabel:'Granularity', targetName:coalesce(gx.name,gx.id), targetFocus:false,
    relType:'HAS_GRANULARITY'
  } END,

  CASE WHEN kx IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:kx.id, targetLabel:'Competence', targetName:coalesce(kx.name,kx.id), targetFocus:false,
    relType:'HAS_COMPETENCE'
  } END,

  CASE WHEN px IS NULL THEN NULL ELSE {
    source:c_used.id, sourceLabel:'Concept', sourceName:coalesce(c_used.name,c_used.id),
    sourceFocus:(c_used.id = c0.id),
    target:px.id, targetLabel:'Phase', targetName:coalesce(px.name,px.id), targetFocus:false,
    relType:'HAS_PHASE'
  } END
] AS edges_real,

// derived edges per slide: focus -> each other concept
[oc IN other_concepts |
  {
    source:c0.id, sourceLabel:'Concept', sourceName:coalesce(c0.name,c0.id), sourceFocus:true,
    target:oc.id, targetLabel:'Concept', targetName:coalesce(oc.name,oc.id), targetFocus:false,
    relType:'CO_OCCURS_WITH'
  }
] AS edges_derived

WITH edges_real + edges_derived AS edges
UNWIND edges AS e
WITH e WHERE e IS NOT NULL
RETURN
  e.source AS source, e.sourceLabel AS sourceLabel, e.sourceName AS sourceName, e.sourceFocus AS sourceFocus,
  e.target AS target, e.targetLabel AS targetLabel, e.targetName AS targetName, e.targetFocus AS targetFocus,
  e.relType AS relType
"""

FOCUS_NODE_QUERY = """
MATCH (c:Concept {id: $id})
RETURN
  toString(c.id) AS id,
  'Concept' AS label,
  coalesce(c.name, toString(c.id)) AS name,
  true AS is_focus
LIMIT 1
"""

# ------------------------------------------------------------
# 5) FETCH TABLES (force-add focus node if missing)
# ------------------------------------------------------------
def fetch_ego(concept_id="", module="", granularity="", competence="", phase="",
             max_slides=60, max_coconcepts_per_slide=80):

    concept_id = (concept_id or "").strip()

    df = cypher_df_driver(driver, EGO_EDGES_QUERY, {
        "concept_id": concept_id,
        "module": (module or ""),
        "granularity": (granularity or ""),
        "competence": (competence or ""),
        "phase": (phase or ""),
        "max_slides": int(max_slides),
        "max_coconcepts_per_slide": int(max_coconcepts_per_slide),
    })

    if df.empty:
        nodes = pd.DataFrame(columns=["id","label","name","is_focus"])
        rels  = pd.DataFrame(columns=["source","target","type"])
    else:
        left = df.rename(columns={
            "source":"id","sourceLabel":"label","sourceName":"name","sourceFocus":"is_focus"
        })[["id","label","name","is_focus"]]

        right = df.rename(columns={
            "target":"id","targetLabel":"label","targetName":"name","targetFocus":"is_focus"
        })[["id","label","name","is_focus"]]

        nodes = pd.concat([left, right], ignore_index=True).dropna(subset=["id"]).drop_duplicates("id")
        nodes["id"] = nodes["id"].astype(str)
        nodes["label"] = nodes["label"].astype(str)
        nodes["name"] = nodes["name"].astype(str)
        nodes["is_focus"] = nodes["is_focus"].apply(_to_bool)

        rels = df.rename(columns={"relType":"type"})[["source","target","type"]].dropna()
        rels["source"] = rels["source"].astype(str)
        rels["target"] = rels["target"].astype(str)
        rels["type"] = rels["type"].astype(str)

    # ensure focus node exists
    if concept_id:
        if nodes.empty or concept_id not in set(nodes["id"].astype(str)):
            fdf = cypher_df_driver(driver, FOCUS_NODE_QUERY, {"id": concept_id})
            if not fdf.empty and "id" in fdf.columns:
                fdf["id"] = fdf["id"].astype(str)
                fdf["label"] = fdf["label"].astype(str)
                fdf["name"] = fdf["name"].astype(str)
                fdf["is_focus"] = fdf["is_focus"].apply(_to_bool)
                nodes = pd.concat([nodes, fdf[["id","label","name","is_focus"]]],
                                  ignore_index=True).drop_duplicates("id")

    return nodes, rels

# ------------------------------------------------------------
# 6) PLOT
# ------------------------------------------------------------
NODE_COLORS = {
    "Concept":"#1F77B4",
    "Module":"#2A9D8F",
    "Slide":"#6C757D",
    "Phase":"#D1495B",
    "Competence":"#E9C46A",
    "Granularity":"#2CA02C",
    "Other":"#B0B0B0",
}
FOCUS_NODE_COLOR = "#7C3AED"
FOCUS_RING_COLOR = "#FBBF24"
COCONCEPT_EDGE_COLOR = "#EF4444"  # red

def plot_graph(nodes, rels, focus_id="", title="Ego Network (focus centered)"):
    focus_id = (focus_id or "").strip()

    if nodes.empty:
        fig = go.Figure()
        fig.update_layout(title="No results (try relaxing filters)")
        return fig

    # label lookup (case-normalized)
    node_label = {}
    for r in nodes.to_dict("records"):
        nid = str(r["id"])
        lab = str(r.get("label", "Other")).strip()
        node_label[nid] = lab

    node_label_norm = {k: (v.strip().lower() if isinstance(v, str) else "other") for k, v in node_label.items()}

    # graph for layout
    G = nx.Graph()
    for r in nodes.to_dict("records"):
        G.add_node(str(r["id"]), **r)

    # keep edge types
    edge_types = {}
    for r in rels.to_dict("records"):
        u = str(r["source"]); v = str(r["target"])
        t = str(r.get("type", ""))
        G.add_edge(u, v)
        edge_types[(u, v)] = t
        edge_types[(v, u)] = t

    focus = focus_id if (focus_id and focus_id in G.nodes) else None

    init_pos = nx.random_layout(G, seed=42)
    if focus:
        init_pos[focus] = np.array([0.0, 0.0])

    pos = nx.spring_layout(
        G, seed=42,
        pos=init_pos,
        fixed=[focus] if focus else None,
        k=2.2 / max(1, (G.number_of_nodes() ** 0.5)),
        iterations=400
    )

    # center focus
    if focus and focus in pos:
        fx, fy = pos[focus]
        for n in pos:
            pos[n] = (pos[n][0] - fx, pos[n][1] - fy)

    # OPTIONAL: highlight slide->concept edges on focus slides
    focus_slides = set()
    coc_edges_slide_to_concept = set()

    if focus and (rels is not None) and (not rels.empty):
        r2 = rels.copy()
        r2["source"] = r2["source"].astype(str)
        r2["target"] = r2["target"].astype(str)
        r2["type"] = r2["type"].astype(str)

        foc_rel = r2[r2["type"] == "FOCUSES_ON"]

        for _, row in foc_rel.iterrows():
            u, v = row["source"], row["target"]
            if u == focus and node_label_norm.get(v, "") == "slide":
                focus_slides.add(v)
            elif v == focus and node_label_norm.get(u, "") == "slide":
                focus_slides.add(u)

        if focus_slides:
            for _, row in foc_rel.iterrows():
                u, v = row["source"], row["target"]
                if u in focus_slides and node_label_norm.get(v, "") == "concept" and v != focus:
                    coc_edges_slide_to_concept.add((u, v))
                elif v in focus_slides and node_label_norm.get(u, "") == "concept" and u != focus:
                    coc_edges_slide_to_concept.add((v, u))

    # precompute derived CO_OCCURS_WITH edges from rels (returned by Cypher now)
    cooccurs_edges = set()
    if rels is not None and not rels.empty:
        rr = rels.copy()
        rr["source"] = rr["source"].astype(str)
        rr["target"] = rr["target"].astype(str)
        rr["type"] = rr["type"].astype(str)
        rr_co = rr[rr["type"] == "CO_OCCURS_WITH"]
        for _, row in rr_co.iterrows():
            cooccurs_edges.add((row["source"], row["target"]))
            cooccurs_edges.add((row["target"], row["source"]))

    # Edge buckets
    bg_x, bg_y = [], []
    focus_x, focus_y = [], []
    coc_slide_x, coc_slide_y = [], []
    coc_virtual_x, coc_virtual_y = [], []

    def _seg(arrx, arry, u, v):
        if u not in pos or v not in pos:
            return
        x0, y0 = pos[u]; x1, y1 = pos[v]
        arrx += [x0, x1, None]
        arry += [y0, y1, None]

    for u, v in G.edges():
        u = str(u); v = str(v)

        is_focus_edge = bool(focus) and (u == focus or v == focus)
        is_coconcept_slide_edge = ((u, v) in coc_edges_slide_to_concept) or ((v, u) in coc_edges_slide_to_concept)
        is_cooccurs = ((u, v) in cooccurs_edges)  # ✅ focus -> other concept derived edge

        if is_cooccurs:
            _seg(coc_virtual_x, coc_virtual_y, u, v)
        elif is_coconcept_slide_edge:
            _seg(coc_slide_x, coc_slide_y, u, v)
        elif is_focus_edge:
            _seg(focus_x, focus_y, u, v)
        else:
            _seg(bg_x, bg_y, u, v)

    edge_bg = go.Scatter(
        x=bg_x, y=bg_y, mode="lines",
        line=dict(width=1, color="#B8BCC2"),
        opacity=0.18, hoverinfo="none",
        name="edges"
    )

    edge_coc_slides = go.Scatter(
        x=coc_slide_x, y=coc_slide_y, mode="lines",
        line=dict(width=2.0, color=COCONCEPT_EDGE_COLOR),
        opacity=0.35, hoverinfo="none",
        name="edges (focus slides → other concepts)"
    )

    edge_cooccurs = go.Scatter(
        x=coc_virtual_x, y=coc_virtual_y, mode="lines",
        line=dict(width=3.6, color=COCONCEPT_EDGE_COLOR),
        opacity=0.95, hoverinfo="none",
        name="edges (FOCUS → other concepts)"
    )

    edge_focus = go.Scatter(
        x=focus_x, y=focus_y, mode="lines",
        line=dict(width=3.2, color="#111827"),
        opacity=0.85, hoverinfo="none",
        name="edges (focus)"
    )

    # Nodes (grouped by label)
    node_traces = []
    by_label = {}
    for n, d in G.nodes(data=True):
        if focus and n == focus:
            continue
        by_label.setdefault(str(d.get("label", "Other")).strip(), []).append((n, d))

    for lab, items in sorted(by_label.items()):
        xs, ys, h = [], [], []
        sizes, cols = [], []
        for n, d in items:
            x, y = pos[n]
            xs.append(x); ys.append(y)
            h.append(
                "<br>".join([
                    f"<b>{d.get('id', n)}</b>",
                    f"type: {d.get('label', '')}",
                    f"name: {d.get('name', '')}",
                ])
            )
            sizes.append(14 if lab.strip().lower() == "concept" else 10)
            cols.append(NODE_COLORS.get(lab, NODE_COLORS["Other"]))

        node_traces.append(go.Scatter(
            x=xs, y=ys, mode="markers",
            marker=dict(size=sizes, color=cols, opacity=0.95, line=dict(width=0)),
            hovertext=h, hoverinfo="text",
            name=f"Node: {lab}",
            showlegend=True
        ))

    # Focus marker
    focus_trace = []
    warning = ""
    if focus_id and (focus is None):
        warning = "⚠️ Selected concept is not present under current filters (try relaxing module/granularity/competence/phase)."
    elif focus:
        fx, fy = pos[focus]
        fd = G.nodes[focus]
        label = str(fd.get("id", focus))
        if len(label) > 70:
            label = label[:67] + "..."

        focus_trace = [
            go.Scatter(
                x=[fx], y=[fy], mode="markers",
                marker=dict(size=70, color=FOCUS_NODE_COLOR, opacity=1.0,
                            line=dict(width=7, color=FOCUS_RING_COLOR)),
                hovertext=[f"<b>{fd.get('id', focus)}</b><br>FOCUS (selected concept)"],
                hoverinfo="text",
                name="FOCUS (selected concept)",
                showlegend=True
            ),
            go.Scatter(
                x=[fx], y=[fy], mode="text",
                text=[label],
                textposition="middle right",
                textfont=dict(size=14),
                hoverinfo="none",
                showlegend=False
            )
        ]

    fig = go.Figure([edge_bg, edge_coc_slides, edge_cooccurs, edge_focus] + node_traces + focus_trace)
    fig.update_layout(
        title=f"{title} — {G.number_of_nodes()} nodes / {G.number_of_edges()} edges",
        hovermode="closest",
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(itemsizing="constant"),
    )

    if warning:
        fig.add_annotation(
            xref="paper", yref="paper",
            x=0.01, y=1.08,
            text=warning,
            showarrow=False,
            font=dict(size=12),
            bgcolor="rgba(255,255,255,0.85)"
        )

    return fig

# ------------------------------------------------------------
# 7) UI (Module/Granularity/Competence/Phase from DB; Concepts from CSV)
# ------------------------------------------------------------
modules = cypher_df_driver(driver, "MATCH (m:Module) RETURN DISTINCT toString(m.id) AS id ORDER BY id")["id"].dropna().tolist()
grans   = cypher_df_driver(driver, "MATCH (g:Granularity) RETURN DISTINCT toString(g.id) AS id ORDER BY id")["id"].dropna().tolist()
comps   = cypher_df_driver(driver, "MATCH (k:Competence) RETURN DISTINCT toString(k.id) AS id ORDER BY id")["id"].dropna().tolist()
phases  = cypher_df_driver(driver, "MATCH (p:Phase) RETURN DISTINCT toString(p.id) AS id ORDER BY id")["id"].dropna().tolist()

module_dd = widgets.Dropdown(options=[""] + modules, description="Module:", layout=widgets.Layout(width="240px"))
gran_dd   = widgets.Dropdown(options=[""] + grans,   description="Granularity:", layout=widgets.Layout(width="240px"))
comp_dd   = widgets.Dropdown(options=[""] + comps,   description="Competence:", layout=widgets.Layout(width="240px"))
phase_dd  = widgets.Dropdown(options=[""] + phases,  description="Phase:", layout=widgets.Layout(width="240px"))

concept_search = widgets.Text(value="", description="Search:", layout=widgets.Layout(width="420px"))
concept_dd = widgets.Dropdown(options=[""] + ALL_CONCEPTS, description="Concept:", layout=widgets.Layout(width="520px"))

slides_sl  = widgets.IntSlider(value=60, min=5, max=400, step=5, description="Max slides:", layout=widgets.Layout(width="320px"))
co_sl      = widgets.IntSlider(value=80, min=1, max=800, step=1, description="Co-concepts/slide:", layout=widgets.Layout(width="360px"))

apply_btn  = widgets.Button(description="Apply", button_style="primary", layout=widgets.Layout(width="120px"))
clear_btn  = widgets.Button(description="Clear", layout=widgets.Layout(width="120px"))
status_lbl = widgets.HTML(value="")
out = widgets.Output()

def update_concept_options(*_):
    q = concept_search.value.strip().lower()
    selected = (concept_dd.value or "").strip()

    if not q:
        opts = [""] + ALL_CONCEPTS
    else:
        filtered = [c for c in ALL_CONCEPTS if q in c.lower()]
        if selected and (selected not in filtered):
            filtered = [selected] + filtered
        opts = [""] + filtered

    concept_dd.options = opts
    if selected and selected not in concept_dd.options:
        concept_dd.value = ""

def refresh(_=None):
    with out:
        clear_output(wait=True)

        status_lbl.value = (
            f"<b>Ego view:</b> concept='{concept_dd.value or '*'}', "
            f"module='{module_dd.value or '*'}', "
            f"granularity='{gran_dd.value or '*'}', "
            f"competence='{comp_dd.value or '*'}', "
            f"phase='{phase_dd.value or '*'}', "
            f"max_slides={slides_sl.value}, "
            f"co_concepts/slide={co_sl.value}"
        )

        nodes, rels = fetch_ego(
            concept_id=concept_dd.value,
            module=module_dd.value,
            granularity=gran_dd.value,
            competence=comp_dd.value,
            phase=phase_dd.value,
            max_slides=slides_sl.value,
            max_coconcepts_per_slide=co_sl.value
        )

        fig = plot_graph(nodes, rels, focus_id=concept_dd.value, title="Ego Network (focus centered)")
        fig.show()

def clear_filters(_=None):
    module_dd.value = ""
    gran_dd.value = ""
    comp_dd.value = ""
    phase_dd.value = ""
    concept_search.value = ""
    concept_dd.value = ""
    slides_sl.value = 60
    co_sl.value = 80
    refresh()

concept_search.observe(update_concept_options, names="value")
apply_btn.on_click(refresh)
clear_btn.on_click(clear_filters)

display(widgets.VBox([
    widgets.HBox([module_dd, gran_dd, comp_dd, phase_dd]),
    widgets.HBox([concept_search, concept_dd]),
    widgets.HBox([slides_sl, co_sl, apply_btn, clear_btn]),
    status_lbl,
    out
]))

update_concept_options()
refresh()


=== Exported graph summary (from CSV) ===
Unique nodes: 568
Unique edges: 1332

Nodes by label:
label
Competence       3
Concept        236
Granularity      3
Module          10
Phase            4
Slide          312
Name: count, dtype: int64

Edges by type:
type
FOCUSES_ON         312
HAS_COMPETENCE     236
HAS_GRANULARITY    236
HAS_PHASE          236
HAS_SLIDE          312
Name: count, dtype: int64

Loaded cypher_fetch_used.cypher (first 400 chars):
// QUERY 1: FETCH_NODES_ALL
MATCH (n)
WHERE n.id IS NOT NULL
RETURN
  n.id AS id,
  labels(n)[0] AS label,
  coalesce(n.name, n.id) AS name,
  coalesce(n.frequency, 0) AS frequency,
  coalesce(n.modules, 0) AS modules,
  coalesce(n.phase_mode, "") AS phase_mode,
  coalesce(n.competence_mode, "") AS competence_mode,
  coalesce(n.granularity_mode, "") AS granularity_mode

// QUERY 2: FETCH_EDGES_ALL
...

